In [10]:
from opencc import OpenCC
from rich.progress import Progress
import os

In [11]:
# all .dict.yaml files in cn_dicts
cn_dict_files = []
for root, dirs, files in os.walk('cn_dicts'):
	for file in files:
		if file.endswith('.dict.yaml'):
			cn_dict_files.append(os.path.join(root, file))

tw_dict_files = []
for root, dirs, files in os.walk('tw_dicts'):
	for file in files:
		if file.endswith('.dict.yaml'):
			tw_dict_files.append(os.path.join(root, file))

In [17]:
def dict_convert(from_dir: str, to_dir: str, convert_method: str):
	from_dict_files = []
	for root, dirs, files in os.walk(from_dir):
		for file in files:
			if file.endswith('.dict.yaml') and not file.endswith('.translated.dict.yaml'):
				from_dict_files.append(os.path.join(root, file))
	
	# translation dict
	translation_dict_file = f"trans_dict_{convert_method}.txt"
	converter = OpenCC(convert_method)
	if not os.path.exists(translation_dict_file):
		with open(translation_dict_file, 'w', encoding="utf-8-sig") as f:
			f.write("")
	translation_dict = {}
	with open(translation_dict_file, 'r', encoding="utf-8-sig") as f:
		for line in f:
			if not line.strip() or line.startswith("#"):
				continue
			key, val = line.strip().split("\t")
			translation_dict[key] = val

	# unchecked
	translation_dict_unchecked = {}

	with Progress() as progress:
		task = progress.add_task("[cyan]Translating...", total=len(from_dict_files))
		for from_dict_file in from_dict_files:
			to_dict_file = from_dict_file.replace(from_dir, to_dir)
			to_dict_file = to_dict_file.replace(".dict.yaml", ".translated.dict.yaml")
			to_dict_lines = []

			with open(from_dict_file, 'r', encoding="utf-8-sig") as f:
				task_single_file = progress.add_task(os.path.basename(from_dict_file), total=os.path.getsize(from_dict_file))

				# header lines
				for line in f:
					to_dict_lines.append(line)
					progress.update(task_single_file, advance=len(line.encode("utf-8")))
					if line == "...\n":
						break
				
				# entries
				for line in f:
					if not line.startswith("#"):  # skip comments
						if "\t" in line:
							key, val = line.split("\t")[0], "\t".join(line.split("\t")[1:])
							if key in translation_dict:
								translation = translation_dict[key]
							elif key in translation_dict_unchecked:
								translation = translation_dict_unchecked[key]
							else:
								translation = converter.convert(key)
								translation_dict_unchecked[key] = translation
							line = translation + "\t" + val
						elif line.strip():
							if line in translation_dict:
								translation = translation_dict[line]
							elif line in translation_dict_unchecked:
								translation = translation_dict_unchecked[line]
							else:
								translation = converter.convert(line)
								translation_dict_unchecked[line] = translation
							line = translation
					to_dict_lines.append(line)
					progress.update(task_single_file, advance=len(line.encode("utf-8")))
			progress.remove_task(task_single_file)

			task_write = progress.add_task("Writing...", total=1)
			with open(to_dict_file, 'w', encoding="utf-8-sig") as f:
				f.writelines(to_dict_lines)
			progress.remove_task(task_write)

			progress.update(task, advance=1)

	with open(translation_dict_file, 'w', encoding="utf-8-sig") as f:
		# first write checked translations
		identical = []
		for key, val in translation_dict.items():
			if key == val:
				identical.append(key)
			else:
				f.write(key + "\t" + val + "\n")
		if identical:
			f.write("\n# Identical translations\n")
			for key in identical:
				f.write(key + "\t" + key + "\n")

		# then write unchecked translations
		if translation_dict_unchecked:
			f.write("\n# Unchecked translations\n")
			identical = []
			for key, val in translation_dict_unchecked.items():
				if key == val:
					identical.append(key)
				else:
					f.write(key + "\t" + val + "\n")
			if identical:
				f.write("\n# Identical unchecked translations\n")
				for key in identical:
					f.write(key + "\t" + key + "\n")

In [18]:
dict_convert("cn_dicts", "tw_dicts", "s2tw")
dict_convert("tw_dicts", "cn_dicts", "t2s")

Output()

Output()